# 2장 1강: AARRR 프레임워크 개요와 단계별 분석 목적 — 실습문제

## 실습 목표

- Ravenstack의 사용자 여정을 AARRR 5단계로 구분할 수 있다.
- 각 단계의 분석 목적과 활용 가능한 데이터를 연결할 수 있다.
- 이벤트 로그의 사용자·행동·시점·속성 컬럼을 구분할 수 있다.
- 분석 질문에 맞게 퍼널·코호트·세그멘테이션 분석을 선택할 수 있다.
- 그룹별 Activation 지표를 비교하여 점검 대상을 찾을 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`
- `ravenstack_feature_usage.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

Ravenstack의 AARRR 단계는 다음과 같이 연결할 수 있습니다.

| 단계 | Ravenstack에서의 의미 | 활용 데이터 예시 |
|---|---|---|
| Acquisition | 고객사가 유입되어 가입함 | 가입일, 유입 경로 |
| Activation | 구독 후 제품 기능을 처음 사용함 | 기능 사용 기록 |
| Retention | 제품 사용과 구독을 계속 유지함 | 반복 사용 기록, 이탈 여부 |
| Referral | 기존 고객이 다른 고객을 추천함 | 추천 전송·추천 가입 이벤트 |
| Revenue | 유료 구독에서 반복 매출이 발생함 | MRR, 요금제 |

> 제공된 데이터에는 고객 추천 행동을 직접 기록한 이벤트가 없습니다. `referral_source`는 고객의 유입 경로이며, 기존 고객이 추천 행동을 했다는 사실을 직접 보여주는 Referral 이벤트와는 구분해야 합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 세 CSV 파일을 각각 `accounts`, `subscriptions`, `feature_usage`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`, `usage_date`를 날짜형으로 변환하세요.
6. `feature_usage`에서 이벤트 식별자인 `usage_id`의 중복 개수를 확인하세요.
7. 기능별 이벤트 기록 수와 `usage_count`의 기초 통계량을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
# 1. 필요한 라이브러리 불러오기
import pandas as pd

# 2. CSV 파일 불러오기
accounts = pd.read_csv("ravenstack_accounts.csv")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")
feature_usage = pd.read_csv("ravenstack_feature_usage.csv")

# 3. 행과 열 개수, 상위 5개 행 확인
print("[accounts]")
print("크기:", accounts.shape)
print(accounts.head())

print("\n[subscriptions]")
print("크기:", subscriptions.shape)
print(subscriptions.head())

print("\n[feature_usage]")
print("크기:", feature_usage.shape)
print(feature_usage.head())

# 4. 분석 대상 컬럼의 자료형과 결측치 개수 확인
print("\n[accounts 자료형]")
print(accounts.dtypes)
print("\n[accounts 결측치]")
print(accounts.isna().sum())

print("\n[subscriptions 자료형]")
print(subscriptions.dtypes)
print("\n[subscriptions 결측치]")
print(subscriptions.isna().sum())

print("\n[feature_usage 자료형]")
print(feature_usage.dtypes)
print("\n[feature_usage 결측치]")
print(feature_usage.isna().sum())

# 5. 날짜형으로 변환
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])

subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])

feature_usage["usage_date"] = pd.to_datetime(feature_usage["usage_date"])

# 6. usage_id 중복 개수 확인
duplicate_usage_id = feature_usage["usage_id"].duplicated().sum()

print("\nusage_id 중복 개수:", duplicate_usage_id)

# 7. 기능별 이벤트 기록 수와 usage_count 기초 통계량 확인
print("\n[기능별 이벤트 기록 수]")
print(feature_usage["feature_name"].value_counts())

print("\n[usage_count 기초 통계량]")
print(feature_usage["usage_count"].describe())

[accounts]
크기: (500, 10)
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  
0       Basic      9     False       False  
1       Basic     18     False        True  
2       Basic      1     False       False  
3       Basic     24      True       False  
4  Enterprise     35     False        True  

[subscriptions]
크기: (5000, 14)
  subscription_id account_id  start_date    end_date   plan_tier  seats  \
0        S-8cec59   A-3c1a3f  2023-12-23  2024-04-12  Enterprise     14   
1        S-0f6f44   A-9b9fe9  2024-06-11         NaN        

---

## 필수 1. Ravenstack의 AARRR 지표 지도 만들기

### 문제 1-1. 데이터에서 확인 가능한 단계별 대표 지표 계산하기

#### 문제 설명

Ravenstack의 세 테이블을 이용하여 AARRR 단계별 대표 지표를 계산하고, 제공된 데이터만으로 직접 측정하기 어려운 단계도 확인하세요.

이번 문제에서는 다음 기준을 사용합니다.

- Acquisition: 가입한 고유 고객사 수
- Activation: 전체 구독 중 기능 사용 기록이 한 번 이상 있는 구독의 비율
- Retention: 전체 구독 중 현재 `churn_flag`가 `False`인 구독의 비율
- Referral: 제공된 데이터로 직접 계산할 수 없음
- Revenue: 종료일이 없고 체험 구독이 아닌 현재 활성 유료 구독의 MRR 합계

> 여기에서 Retention은 학습을 위한 간단한 대리 지표입니다. 특정 기간 후 다시 사용한 비율을 계산한 정식 리텐션과는 다릅니다.

#### 요구사항

1. 가입한 고유 고객사 수를 `acquired_accounts`로 계산하세요.
2. 기능 사용 기록이 있는 고유 구독 수를 전체 구독 수로 나누어 `activation_rate`를 계산하세요.
3. `churn_flag`가 `False`인 구독의 비율을 `non_churned_rate`로 계산하세요.
4. 종료일이 없고 체험 구독이 아닌 구독의 MRR 합계를 `current_paid_mrr`로 계산하세요.
5. 네 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 각 지표를 AARRR 단계와 연결하고 분석 목적을 설명하세요.
7. Referral 단계를 직접 측정하려면 어떤 이벤트가 추가로 필요한지 제안하세요.

#### 해석 질문

**Q1.** 기능 사용 기록이 있는 구독 비율은 어느 AARRR 단계와 연결되나요?  
**Q2.** `non_churned_rate`를 정식 리텐션율과 동일하게 보면 안 되는 이유는 무엇인가요?  
**Q3.** `referral_source`만으로 Referral 행동을 직접 측정할 수 있나요?  
**Q4.** MRR은 어느 AARRR 단계와 연결되나요?

#### 제출 결과

- AARRR 단계별 대표 지표 계산 코드와 결과
- 각 지표의 단계 및 분석 목적
- Referral 측정에 필요한 이벤트 제안
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.
# 1. Acquisition
acquired_accounts = accounts["account_id"].nunique()

# 2. Activation
activated_subscriptions = feature_usage["subscription_id"].nunique()
total_subscriptions = subscriptions["subscription_id"].nunique()

activation_rate = activated_subscriptions / total_subscriptions

# 3. Retention
non_churned_rate = (subscriptions["churn_flag"] == False).mean()

# 4. Revenue
active_paid_mask = (
    subscriptions["end_date"].isna()
    & (subscriptions["is_trial"] == False)
)

current_paid_mrr = subscriptions.loc[
    active_paid_mask, "mrr_amount"
].sum()


# 5. 결과 출력
print(f"Acquisition - 가입 고유 고객사 수: {acquired_accounts:,}")
print(f"Activation - 기능 사용 구독 비율: {activation_rate:.2%}")
print(f"Retention - 비이탈 구독 비율: {non_churned_rate:.2%}")
print("Referral - 제공된 데이터로 직접 계산할 수 없음")
print(f"Revenue - 현재 활성 유료 구독 MRR: {current_paid_mrr:,.2f}")

Acquisition - 가입 고유 고객사 수: 500
Activation - 기능 사용 구독 비율: 99.34%
Retention - 비이탈 구독 비율: 90.28%
Referral - 제공된 데이터로 직접 계산할 수 없음
Revenue - 현재 활성 유료 구독 MRR: 10,159,608.00


### 필수 1 답변 작성란

**Q1.** 기능 사용 기록이 있는 구독 비율은 어느 AARRR 단계와 연결되나요?
  - Activation 단계, 구독한 고객이 제품의 기능을 실제로 사용하여 핵심 가치를 경험했는지를 확인하기 때문

**Q2.** `non_churned_rate`를 정식 리텐션율과 동일하게 보면 안 되는 이유는 무엇인가요?
 - 바이탈 상태만 나타나고 동일한 사용자 그룹이 가입 후 언제 사용했고 다시 사용했는지를 시간에 따라 추적하지 않아서

**Q3.** `referral_source`만으로 Referral 행동을 직접 측정할 수 있나요? 
 - 어렵다. referral_source는 고객이 어떤 경로로 들어왔는지 보여 주지만 이 고객이 다른 고객을 추천해 주거나 유도했다는 행동 자체는 보여 주지 않는다.
 - 이벤트성 변수가 추가적으로 필요하다.

**Q4.** MRR은 어느 AARRR 단계와 연결되나요?
 - Revenue 단계와 연결, 유료 구독에서 발생하는 월간 

---

## 필수 2. 요금제별 Activation 지표 비교하기

### 문제 2-1. 2024년 12월 제품 활성률을 요금제별로 비교하기

#### 문제 설명

프로덕트팀은 다음 질문에 답하려고 합니다.

> 2024년 12월에 어떤 요금제의 제품 활성률이 상대적으로 낮은가?

이 질문은 사용자를 요금제별로 나누어 비교하므로 **세그멘테이션 분석**에 해당하며, AARRR 중 Activation 단계를 살펴봅니다.

이번 실습에서 제품 활성률은 다음과 같이 정의합니다.

> 12월 제품 활성 유료 구독 수 ÷ 12월 이용 가능 유료 구독 수

#### 요구사항

1. 2024년 12월의 시작일과 마지막 날을 설정하세요.
2. 12월에 이용 가능한 유료 구독을 `eligible_dec`에 저장하세요.
3. 12월에 기능을 사용한 구독 ID를 `active_subscription_ids`로 준비하세요.
4. `eligible_dec`에 `is_product_active` 컬럼을 만드세요.
5. `plan_tier`별로 다음 값을 집계하여 `plan_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
6. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
7. 가장 낮은 요금제를 찾아 개선을 위해 추가로 확인할 내용을 제안하세요.
8. 이 질문에 세그멘테이션 분석이 적절한 이유를 설명하세요.

#### 해석 질문

**Q1.** 이 분석은 AARRR 중 어느 단계에 해당하나요?  
**Q2.** 어떤 분석 기법을 사용했으며, 그 이유는 무엇인가요?  
**Q3.** 제품 활성률이 가장 낮은 요금제는 무엇인가요?  
**Q4.** 해당 요금제에서 추가로 어떤 데이터를 확인하면 좋을까요?

#### 제출 결과

- `plan_activation` 집계 코드와 결과
- AARRR 단계와 분석 기법
- 우선 점검할 요금제
- 추가 확인 데이터
- Q1~Q4 답변

In [4]:
# 필수 2 코드를 작성하세요.
dec_start = pd.to_datetime("2024-12-01")
dec_end = pd.to_datetime("2024-12-31")

eligible_dec = subscriptions.loc[
    (subscriptions["is_trial"] == False)
    & (subscriptions["start_date"] <= dec_end)
    & (
        subscriptions["end_date"].isna()
        | (subscriptions["end_date"] >= dec_start)
    )
].copy()

dec_usage = feature_usage.loc[
    (feature_usage["usage_date"] >= dec_start)
    & (feature_usage["usage_date"] <= dec_end)
]

active_subscription_ids = dec_usage["subscription_id"].unique()

eligible_dec["is_product_active"] = eligible_dec[
    "subscription_id"
].isin(active_subscription_ids)

plan_activation = eligible_dec.groupby("plan_tier").agg(
    eligible_paid_subscriptions=("subscription_id", "count"),
    active_paid_subscriptions=("is_product_active", "sum"),
    activation_rate=("is_product_active", "mean")
)

plan_activation = plan_activation.sort_values(
    "activation_rate",
    ascending=True
)

# 활성률이 낮은 순서로 정렬
plan_activation = plan_activation.sort_values(
    "activation_rate",
    ascending=True
)

# 가장 낮은 요금제와 활성률
lowest_plan = plan_activation.index[0]
lowest_rate = plan_activation.loc[lowest_plan, "activation_rate"]

# 출력용 데이터 만들기
plan_activation_display = plan_activation.copy()

plan_activation_display["activation_rate"] = (
    plan_activation_display["activation_rate"] * 100
).round(2).astype(str) + "%"

display(plan_activation_display)

print(f"\n제품 활성률이 가장 낮은 요금제: {lowest_plan}")
print(f"해당 요금제의 제품 활성률: {lowest_rate:.2%}")
print("AARRR 단계: Activation")
print("분석 기법: 세그멘테이션 분석")

,eligible_paid_subscriptions,active_paid_subscriptions,activation_rate
plan_tier,,,
Pro,1342,260,19.37%
Basic,1267,247,19.49%
Enterprise,1372,281,20.48%



제품 활성률이 가장 낮은 요금제: Pro
해당 요금제의 제품 활성률: 19.37%
AARRR 단계: Activation
분석 기법: 세그멘테이션 분석


### 필수 2 답변 작성란

**Q1.** 이 분석은 AARRR 중 어느 단계에 해당하나요?
  - Activation 단계, 유료 구독자가 12월에 제품을 실제로 사용했는지를 확인하려고 했다.

**Q2.** 어떤 분석 기법을 사용했으며, 그 이유는 무엇인가요?
  - 세그멘테이션 분석을 사용, 요금제별로 구독을 나누어 제품의 활성률 차이를 비교했다.

**Q3.** 제품 활성률이 가장 낮은 요금제는 무엇인가요?
  - Pro, 19.37%로 가장 낮은 제품 활성률을 보였다.

**Q4.** 해당 요금제에서 추가로 어떤 데이터를 확인하면 좋을까요?
  - 기능별 사용률, 오류 발생률, 기능 사용 시간, 고객 문의 등 추가...?

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 결제 주기별 Activation 지표 비교하기

#### 문제 설명

프로덕트팀은 2024년 12월 제품 활성률이 월간 결제 구독과 연간 결제 구독에서 다르게 나타나는지 확인하려고 합니다.

> 이 과제는 필수 2의 요금제별 세그멘테이션을 `billing_frequency`에 동일하게 적용하는 문제입니다.

#### 요구사항

1. 필수 2에서 만든 `eligible_dec`과 `is_product_active`를 사용하세요.
2. `billing_frequency`별로 다음 값을 집계하여 `billing_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
3. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
4. 제품 활성률이 더 낮은 결제 주기를 찾으세요.
5. 해당 그룹을 우선 점검 대상으로 정하고 추가로 확인할 내용을 한 가지 제안하세요.
  - 제품 활성률이 낮은 결제 주기의 고객이 어떤 기능을 적게 사용하는지 확인한다.
6. 이 분석의 AARRR 단계와 분석 기법을 설명하세요.
  - Activation
  - 세그멘테이션 분석: 사용자를 특정 특성이나 행동 기준으로 여러 집단으로 나누어 분석하는 방법

#### 해석 질문

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  
**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?
**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

#### 제출 결과

- `billing_activation` 집계 코드와 결과
- 우선 점검할 결제 주기
- 추가 확인 내용
- AARRR 단계와 분석 기법
- Q1~Q4 답변

In [5]:
# 과제 1 코드를 작성하세요.
billing_activation = eligible_dec.groupby("billing_frequency").agg(
    eligible_paid_subscriptions=("subscription_id", "count"),
    active_paid_subscriptions=("is_product_active", "sum"),
    activation_rate=("is_product_active", "mean")
)

billing_activation = billing_activation.sort_values(
    "activation_rate",
    ascending=True
)

lowest_billing = billing_activation.index[0]
lowest_rate = billing_activation.loc[
    lowest_billing, "activation_rate"
]

billing_activation_display = billing_activation.copy()

billing_activation_display["activation_rate"] = (
    billing_activation_display["activation_rate"] * 100
).round(2).astype(str) + "%"

print("[결제 주기별 2024년 12월 제품 활성률]")
print(billing_activation_display)

print(f"\n제품 활성률이 더 낮은 결제 주기: {lowest_billing}")
print(f"해당 결제 주기의 제품 활성률: {lowest_rate:.2%}")
print("AARRR 단계: Activation")
print("분석 기법: 세그멘테이션 분석")

[결제 주기별 2024년 12월 제품 활성률]
                   eligible_paid_subscriptions  active_paid_subscriptions  \
billing_frequency                                                           
annual                                    1968                        385   
monthly                                   2013                        403   

                  activation_rate  
billing_frequency                  
annual                     19.56%  
monthly                    20.02%  

제품 활성률이 더 낮은 결제 주기: annual
해당 결제 주기의 제품 활성률: 19.56%
AARRR 단계: Activation
분석 기법: 세그멘테이션 분석


### 과제 1 답변 작성란

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?
  - 19.56%로 annual이다.

**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
  - Activation 단계
  - 결제 주기라는 속성으로 구독을 나누어 비교했으므로 세그멘테이션 분석이다.

**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?
  - 안 된다. 활성률 차이는 0.46%p이지만 차이만으로 원인을 판단할 수 없으며 요금제나 계정 규모 등 다른 특성이 함께 영향을 줄 수 있다.
  - 결제 주기는 고객이 직접 선택한 것이라 요금제, 계정 규모 같은 다른 특성이 함께 섞여 있을 수 있다.
  - 원인을 판단하려면 결제 주기별 요금제 구성, 기능별 사용량, 오류 수 등을 추가로 확인해야 한다.

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.
| 분석 | 계산/분류 원리 | 분석 질문 | 관련 AARRR |
|---|---|---|---|
| 퍼널 분석 | 전환율 = 다음 단계 사용자 수 ÷ 현재 단계 사용자 수<br>이탈률 = 100% - 전환율 | 어디에서 이탈하는가? | 여러 단계에 활용 가능 |
| 코호트 리텐션 분석 | 같은 시작 기준의 사용자를 묶고 일정 기간 후 다시 활동한 비율 계산 | 시간이 지나도 유지되는가? | Retention |
| 세그멘테이션 분석 | 유입 경로, 결제 주기 등 특성별로 사용자를 나누어 비교 | 어떤 그룹에서 차이가 나타나는가? | 여러 단계에 활용 가능 |
- 퍼널: 현재 단계 사용자 수가 분모이다.
- 코호트: 시작 기준과 경과 기간이 중요하다.
- 세그멘테이션: 사용자 특성이나 행동 기준으로 그룹을 나눈다.

---

## 실습 마무리

아래 질문에 답하세요.

1. Ravenstack의 사용자 여정을 어떤 AARRR 단계로 나누었나요?
 - 고객사 가입은 Acquisition, 첫 기능 사용은 Acitivation, 반복 사용과 구독 유지는 Retention
   고객 추천 Referral, 유료 매출 구독은 Revenue로 나눔

2. 각 단계는 어떤 분석 질문에 답하기 위해 사용했나요?
 - 어디서 고객을 확보했는지, 핵심 기능을 사용했는지, 계속 사용하는지, 다른 고객에게도 추천했는지, 실제 매출이 발생했는지 확인

3. 제공된 데이터에서 직접 측정하기 어려운 단계는 무엇이었나요?
 - Referral 단계, 추천이라는 컬럼이 존재하지 않음.

4. 요금제와 결제 주기별 비교에는 어떤 분석 기법을 사용했나요?
 - 사용자 특성에 따라 그룹을 나누어 비교하는 세그멘테이션 분석 사용

5. 분석 결과를 근거로 어떤 대상을 우선 점검했나요?
 - 요금제에서는 Pro 결제 주기에서는 annual 그룹의 제품 활성률이 낮아 우선 점검 대상으로 정했다.